In [1]:
import platform
import sys
import numpy as np
import pandas as pd
import scipy
import sklearn
import matplotlib

print(f"Python       {sys.version.split()[0]} on {platform.system()}")
for lib in (np, pd, scipy, sklearn, matplotlib):
    print(f"{lib.__name__:<12} {lib.__version__}")

# Optional extras. The notebooks fall back gracefully if these are absent.
for name in ("seaborn", "statsmodels"):
    try:
        mod = __import__(name)
        print(f"{name:<12} {mod.__version__}")
    except ImportError:
        print(f"{name:<12} not installed (optional)")

Python       3.11.16 on Windows
numpy        2.4.6
pandas       3.0.5
scipy        1.17.1
sklearn      1.9.0
matplotlib   3.11.1
seaborn      0.13.2
statsmodels  0.15.0


In [2]:
from pathlib import Path
import matplotlib.pyplot as plt

ROOT = Path.cwd()
# data files may sit in ./data or right next to the notebook
DATA = next((p for p in (ROOT / "data", ROOT) if (p / "anggota.csv").exists()), ROOT / "data")
pd.set_option("display.width", 110)
pd.set_option("display.max_columns", 30)
print("data folder:", DATA)

data folder: c:\Users\ahmad\dev-resources\S2\data-science\project-pasca-uts\data


In [4]:
SENTINELS = ["", "NA", "N/A", "-", "null", "NULL", "unknown", "-999"]

rawDesa = pd.read_excel(
    DATA / "desa.xlsx",
    dtype={"kode": "string"},             # preserve leading zeros
    na_values=SENTINELS,                # declare the folk sentinels
    keep_default_na=True,
    parse_dates=["dihitung_pada"],
    # dayfirst=True,                      # 03/04/2023 is 3 April
)
print("shape:", rawDesa.shape)
print(rawDesa.dtypes.to_string())
rawDesa.head(5)

shape: (79842, 33)
kode                              string
nama                                 str
tipe                                 str
kecamatan                            str
kabupaten                            str
provinsi                             str
luas_desa                        float64
jumlah_persil                      int64
luas_persil                        int64
hak_milik_jumlah                   int64
hak_milik_luas                     int64
hgb_jumlah                         int64
hgb_luas                           int64
hgu_jumlah                         int64
hgu_luas                           int64
hak_pakai_jumlah                   int64
hak_pakai_luas                     int64
hak_pengelolaan_jumlah             int64
hak_pengelolaan_luas               int64
hak_wakaf_jumlah                   int64
hak_wakaf_luas                     int64
kosong_jumlah                      int64
kosong_luas                        int64
lainnya_jumlah                     int

,kode,nama,tipe,kecamatan,kabupaten,provinsi,luas_desa,jumlah_persil,luas_persil,hak_milik_jumlah,hak_milik_luas,hgb_jumlah,hgb_luas,hgu_jumlah,hgu_luas,...,hak_pengelolaan_luas,hak_wakaf_jumlah,hak_wakaf_luas,kosong_jumlah,kosong_luas,lainnya_jumlah,lainnya_luas,tahun_min,tahun_median,tahun_max,rdtr_luas,banjir_rendah,banjir_sedang,banjir_tinggi,dihitung_pada
0,1010101,KOTA BARU,Kelurahan,Kuta Alam,Banda Aceh,Aceh,593698.7,313,679003,219,95859,1,6125,0,0,...,0,0,0,46,181137,0,0,1972.0,2006.0,2026.0,593686.4,22987.2,130725.3,439986.2,2026-10-17 15:12:38
1,1010102,BANDAR BARU,Kelurahan,Kuta Alam,Banda Aceh,Aceh,1139905.6,797,1016060,668,294883,20,8270,0,0,...,0,5,1299,43,84498,0,0,1972.0,2004.0,2026.0,1139905.6,0.0,815590.7,324315.3,2026-10-17 15:12:38
2,1010103,BEURAWE,Kelurahan,Kuta Alam,Banda Aceh,Aceh,799736.9,1601,737670,1455,455608,19,27576,0,0,...,0,2,472,93,190819,0,0,1974.0,2012.0,2026.0,799736.9,0.0,350871.9,448864.9,2026-10-17 15:12:38
3,1010104,KUTA ALAM,Kelurahan,Kuta Alam,Banda Aceh,Aceh,511125.6,576,553700,417,86916,38,25835,0,0,...,2572,2,199,76,132973,0,0,1964.0,2003.0,2026.0,511112.5,0.0,501677.5,9448.1,2026-10-17 15:12:38
4,1010105,KEURAMAT,Kelurahan,Kuta Alam,Banda Aceh,Aceh,326105.0,932,336401,792,174164,18,12373,0,0,...,0,3,1309,79,52816,0,0,1974.0,2001.0,2026.0,326105.0,0.0,326105.0,0.0,2026-10-17 15:12:38


In [5]:
import pandas as pd

def profile(df):
    out = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "n_missing": df.isna().sum(),
        "pct_missing": (df.isna().mean() * 100).round(1),
        "n_unique": df.nunique(dropna=True),
        "example": [
            df[c].dropna().sample(n=min(2, df[c].notna().sum())).tolist() 
            if df[c].notna().any() else []
            for c in df.columns
        ],
    })
    return out.sort_values("pct_missing", ascending=False)

In [6]:
profile(rawDesa)

,dtype,n_missing,pct_missing,n_unique,example
tahun_max,float64,2670,3.3,61,"[2024.0, 2026.0]"
tahun_median,float64,2670,3.3,62,"[2023.0, 2019.0]"
tahun_min,float64,2670,3.3,67,"[1986.0, 1963.0]"
kode,string,0,0.0,79842,"[20030519, 11301008]"
nama,str,1,0.0,63428,"[Mokekuku, Liang]"
provinsi,str,0,0.0,34,"[Papua, Jawa Timur]"
tipe,str,0,0.0,2,"[Desa, Desa]"
kecamatan,str,1,0.0,6732,"[CIKEUSAL, CIRUAS]"
kabupaten,str,0,0.0,487,"[Labuhan Batu, Ogan Komering Ulu]"
hak_milik_jumlah,int64,0,0.0,5826,"[958, 0]"


# Relationship inference on `rawDesa`

Continuation of the notebook. Plan: **(1)** structural relations, **(2)** size-neutral features, **(3)** robust prep.
Steps 4-8 (association matrices, latent structure, within-group checks, ...) will build on the objects created here.

## Step 1: Structural relations

**Goal:** find the relationships that exist *by construction* (identities, inequalities, null patterns, code hierarchy) before looking for statistical ones.

**Why first:** columns that are computed from each other correlate perfectly, and that is not insight. Knowing them up front tells us which columns are redundant and which "discoveries" are fake.

In [7]:
# --- Working copy + column groups -----------------------------------------------------
desa = rawDesa.copy()      # rawDesa stays untouched; everything below works on `desa`

RIGHTS = ["hak_milik", "hgb", "hgu", "hak_pakai", "hak_pengelolaan", "hak_wakaf", "kosong", "lainnya"]
CNT = [f"{r}_jumlah" for r in RIGHTS]      # parcel counts per right
ARE = [f"{r}_luas" for r in RIGHTS]        # parcel areas per right
FLOOD = ["banjir_rendah", "banjir_sedang", "banjir_tinggi"]
YEARS = ["tahun_min", "tahun_median", "tahun_max"]

# `kode` was read as an integer, so its leading zero is gone (1010101 is really 01010101).
# Restore the 8-digit code: PP (province) KK (kab/kota) CC (kecamatan) DD (village)
desa["kode8"] = desa["kode"].astype("int64").astype(str).str.zfill(8)
desa["prov_code"] = desa["kode8"].str[:2]
desa["kab_code"] = desa["kode8"].str[:4]
desa["kec_code"] = desa["kode8"].str[:6]

print("kode length before padding:", desa["kode"].astype(str).str.len().value_counts().to_dict())
print("kode length after padding :", desa["kode8"].str.len().value_counts().to_dict())
desa[["kode", "kode8", "prov_code", "kab_code", "kec_code", "nama", "kabupaten", "provinsi"]].head()

kode length before padding: {8: 54442, 7: 25400}
kode length after padding : {8: 79842}


,kode,kode8,prov_code,kab_code,kec_code,nama,kabupaten,provinsi
0,1010101,01010101,01,0101,010101,KOTA BARU,Banda Aceh,Aceh
1,1010102,01010102,01,0101,010101,BANDAR BARU,Banda Aceh,Aceh
2,1010103,01010103,01,0101,010101,BEURAWE,Banda Aceh,Aceh
3,1010104,01010104,01,0101,010101,KUTA ALAM,Banda Aceh,Aceh
4,1010105,01010105,01,0101,010101,KEURAMAT,Banda Aceh,Aceh


In [8]:
# --- Rule book: every structural relation we test gets one row -------------------------
RULES = []     # re-running this cell starts a fresh rule book

def add_rule(name, kind, violated, scope=None, note=""):
    """violated: boolean Series, True where the rule is broken.
       scope: boolean Series of the rows the rule applies to (default: all rows)."""
    scope = pd.Series(True, index=desa.index) if scope is None else scope
    n, v = int(scope.sum()), int((violated & scope).sum())
    RULES.append({"rule": name, "kind": kind, "n_checked": n, "n_violated": v,
                  "pct_violated": round(100 * v / n, 3) if n else np.nan, "note": note})

# 1) Identities: the 8 rights should be a complete partition of a village's parcels
add_rule("sum(8 rights _jumlah) == jumlah_persil", "identity", desa[CNT].sum(axis=1) != desa["jumlah_persil"])
add_rule("sum(8 rights _luas) == luas_persil", "identity", desa[ARE].sum(axis=1) != desa["luas_persil"])

# 2) Zero coupling: a right has zero parcels exactly when it has zero area
for r in RIGHTS:
    add_rule(f"{r}: jumlah == 0  <=>  luas == 0", "identity", (desa[f"{r}_jumlah"] == 0) != (desa[f"{r}_luas"] == 0))
add_rule("jumlah_persil == 0  <=>  luas_persil == 0", "identity", (desa["jumlah_persil"] == 0) != (desa["luas_persil"] == 0))

# 3) Inequalities: one area can never exceed the area that contains it
TOL = 1.0001     # flood / RDTR areas are rounded to 0.1 m2, so allow 0.01% slack
fsum = desa[FLOOD].sum(axis=1)
add_rule("flood(low+mid+high) <= luas_desa", "inequality", fsum > desa["luas_desa"] * TOL)
add_rule("rdtr_luas <= luas_desa", "inequality", desa["rdtr_luas"] > desa["luas_desa"] * TOL)
add_rule("luas_persil <= luas_desa", "inequality (soft)", desa["luas_persil"] > desa["luas_desa"],
         scope=desa["luas_desa"] > 0, note="parcels can overlap / boundaries may not match; rows with luas_desa == 0 are skipped")
has_year = desa[YEARS].notna().all(axis=1)
add_rule("tahun_min <= tahun_median <= tahun_max", "inequality",
         (desa["tahun_min"] > desa["tahun_median"]) | (desa["tahun_median"] > desa["tahun_max"]), scope=has_year)

# 4) Null patterns: when are the year columns empty?
titled = desa["jumlah_persil"] - desa["kosong_jumlah"]       # parcels that carry some right (assumes kosong = no right)
yr_null = desa[YEARS].isna().all(axis=1)
add_rule("tahun_* are null together", "null pattern", desa[YEARS].isna().any(axis=1) & ~yr_null)
add_rule("year present  =>  at least 1 titled parcel", "implication", titled == 0, scope=has_year)
add_rule("no titled parcel  =>  year is null", "implication", ~yr_null, scope=(titled == 0))
add_rule("year is null  =>  no titled parcel", "implication", titled > 0, scope=yr_null)

rules = pd.DataFrame(RULES)
rules

,rule,kind,n_checked,n_violated,pct_violated,note
0,sum(8 rights _jumlah) == jumlah_persil,identity,79842,0,0.000,
1,sum(8 rights _luas) == luas_persil,identity,79842,0,0.000,
2,hak_milik: jumlah == 0 <=> luas == 0,identity,79842,0,0.000,
3,hgb: jumlah == 0 <=> luas == 0,identity,79842,0,0.000,
4,hgu: jumlah == 0 <=> luas == 0,identity,79842,0,0.000,
5,hak_pakai: jumlah == 0 <=> luas == 0,identity,79842,0,0.000,
6,hak_pengelolaan: jumlah == 0 <=> luas == 0,identity,79842,0,0.000,
7,hak_wakaf: jumlah == 0 <=> luas == 0,identity,79842,0,0.000,
8,kosong: jumlah == 0 <=> luas == 0,identity,79842,6,0.008,
9,lainnya: jumlah == 0 <=> luas == 0,identity,79842,0,0.000,


In [9]:
# --- The rows that break the "almost always true" rules ---------------------------------
# (a) year is null although titled parcels exist
display(desa.loc[yr_null & (titled > 0), ["kode8", "nama", "kabupaten", "jumlah_persil", "kosong_jumlah", "hak_milik_jumlah", "hgb_jumlah"]])

# (b) kosong has parcels but zero area
display(desa.loc[(desa["kosong_jumlah"] > 0) & (desa["kosong_luas"] == 0), ["kode8", "nama", "kabupaten", "jumlah_persil", "kosong_jumlah", "kosong_luas"]])

# (c) how the year columns relate to the titled count
pd.crosstab(yr_null.rename("year is null"), (titled == 0).rename("no titled parcel"))

,kode8,nama,kabupaten,jumlah_persil,kosong_jumlah,hak_milik_jumlah,hgb_jumlah
8426,02100885,Suro Dingin,Tapanuli Selatan,1,0,1,0
9689,02150606,Siopatbahal,Tapanuli Utara,5,3,0,0
17176,05011302,Kec. Kampar,Pekanbaru,1,0,1,0
28164,10160916,Legok,SUMEDANG,3,2,1,0
31667,11019912,KINTELAN,SEMARANG,104,103,0,1
78849,33021105,Maralol,Sorong,2,0,0,0


,kode8,nama,kabupaten,jumlah_persil,kosong_jumlah,kosong_luas
11104,02230726,SIROFI,Nias Selatan,77,1,0
16550,04170304,BANDING AGUNG,Ogan Komering Ulu Selatan,184,1,0
16703,04171308,PENANGGUNGAN,Ogan Komering Ulu Selatan,284,2,0
50883,14120219,Nanga Menunuk,Melawi,274,1,0
64445,21072402,TALUMBINGA,Konawe Selatan,13,1,0
69835,24140709,BELABAJA,Lembata,210,1,0


no titled parcel,False,True
year is null,,
False,77172,0
True,6,2664


In [10]:
# --- Hierarchy: does the 8-digit code tell the same story as the names? ----------------
# Names are written inconsistently (KUTA ALAM vs Kuta Alam, double spaces), so also test a normalized version.
def norm(s):
    return s.str.upper().str.replace(r"\s+", " ", regex=True).str.strip()

for c in ["provinsi", "kabupaten", "kecamatan", "nama"]:
    desa[f"{c}_n"] = norm(desa[c])

def hierarchy_check(code_col, name_col, parent_col=None):
    """code -> name should be 1:1. Within the same parent, a name should map to a single code."""
    row = {"code": code_col, "name": name_col, "n_codes": desa[code_col].nunique(),
           "n_names_raw": desa[name_col].nunique(), "n_names_norm": desa[f"{name_col}_n"].nunique()}
    for tag, col in (("raw", name_col), ("norm", f"{name_col}_n")):
        row[f"codes_with_>1_name_{tag}"] = int((desa.groupby(code_col)[col].nunique() > 1).sum())
    keys = [parent_col, f"{name_col}_n"] if parent_col else [f"{name_col}_n"]
    row["names_with_>1_code_same_parent"] = int((desa.groupby(keys)[code_col].nunique() > 1).sum())
    return row

hier = pd.DataFrame([
    hierarchy_check("prov_code", "provinsi"),
    hierarchy_check("kab_code", "kabupaten", "prov_code"),
    hierarchy_check("kec_code", "kecamatan", "kab_code"),
])
display(hier)

# Province code vs province name: which rows carry a code that belongs to another province?
modal = desa.groupby("provinsi")["prov_code"].agg(lambda s: s.value_counts().idxmax())
desa["prov_code_mismatch"] = desa["prov_code"] != desa["provinsi"].map(modal)
print("rows where prov_code disagrees with provinsi:", int(desa["prov_code_mismatch"].sum()))
display(desa.loc[desa["prov_code_mismatch"], ["kode8", "prov_code", "nama", "kabupaten", "provinsi"]].assign(
    expected_code=lambda x: x["provinsi"].map(modal)))

# Kabupaten names that hide two different kab/kota codes inside the SAME province (Kabupaten X vs Kota X)
multi = desa.groupby(["prov_code", "kabupaten_n"])["kab_code"].agg(lambda s: sorted(s.unique()))
multi = multi[multi.str.len() > 1]
print("kabupaten names with more than one kab_code in the same province:", len(multi))
display(multi.head(12))

# ...and the opposite problem: one kab_code shared by two different kabupaten names
names_per_code = desa.groupby("kab_code")["kabupaten_n"].agg(lambda s: sorted(s.dropna().unique()))
shared = names_per_code[names_per_code.str.len() > 1].to_frame("kabupaten_names")
shared["n_villages"] = desa.groupby("kab_code").size()
print("kab_code shared by more than one kabupaten name:", len(shared))
shared

,code,name,n_codes,n_names_raw,n_names_norm,codes_with_>1_name_raw,codes_with_>1_name_norm,names_with_>1_code_same_parent
0,prov_code,provinsi,34,34,34,3,3,4
1,kab_code,kabupaten,508,487,478,10,10,33
2,kec_code,kecamatan,6941,6732,6575,75,70,42


rows where prov_code disagrees with provinsi: 17


,kode8,prov_code,nama,kabupaten,provinsi,expected_code
18083,05061009,05,TUAPAYA,Bintan,Kepulauan Riau,32
41538,12120320,12,SIGODANG TUA,Toba,Sumatera Utara,02
53437,16081005,16,PA URUD,Nunukan,Kalimantan Utara,34
53438,16084414,16,SALANG,Nunukan,Kalimantan Utara,34
53601,16110110,16,Muara Pinang Lama,Empat Lawang,Sumatera Selatan,04
53615,16110219,16,PAGAR TENGAH,Empat Lawang,Sumatera Selatan,04
53616,16110230,16,PENDOPO,Empat Lawang,Sumatera Selatan,04
53636,16110408,16,RANTAU TENANG,Empat Lawang,Sumatera Selatan,04
53638,16110413,16,TANJUNG KUPANG,Empat Lawang,Sumatera Selatan,04
53639,16110417,16,PASAR TEBING TINGGI,Empat Lawang,Sumatera Selatan,04


kabupaten names with more than one kab_code in the same province: 33


prov_code  kabupaten_n     
02         PADANG SIDEMPUAN          [0210, 0220]
03         SOLOK                     [0308, 0309]
           SOLOK SELATAN             [0308, 0319]
04         EMPAT LAWANG              [0405, 0418]
05         KUANTAN SINGINGI          [0503, 0514]
           ROKAN HILIR               [0510, 0511]
08         MESUJI                    [0806, 0814]
10         BANDUNG                   [1014, 1015]
           BEKASI                    [1005, 1026]
           BOGOR               [1009, 1010, 1033]
           CIREBON                   [1020, 1021]
           SUKABUMI                  [1011, 1012]
Name: kab_code, dtype: object

kab_code shared by more than one kabupaten name: 10


,kabupaten_names,n_villages
kab_code,,
0210,"[PADANG SIDEMPUAN, TAPANULI SELATAN]",704
0308,"[SOLOK, SOLOK SELATAN]",73
0405,"[EMPAT LAWANG, LAHAT]",384
0503,"[INDRAGIRI HULU, KUANTAN SINGINGI]",206
0511,"[ROKAN HILIR, SIAK]",133
0806,"[MESUJI, TULANG BAWANG]",158
1212,"[JOMBANG, TOBA]",307
1611,"[EMPAT LAWANG, KUTAI BARAT]",232
2102,"[KOLAKA, KOLAKA TIMUR]",135


In [11]:
# --- Same-name villages inside one kecamatan (name is not a key) -------------------------
dup = desa[desa.duplicated(["kec_code", "nama_n"], keep=False)].sort_values(["kec_code", "nama_n"])
print("rows involved:", len(dup), "| groups:", dup.groupby(["kec_code", "nama_n"]).ngroups)
dup[["kode8", "nama", "tipe", "kecamatan", "kabupaten", "luas_desa", "jumlah_persil"]].head(10)

rows involved: 85 | groups: 42


,kode8,nama,tipe,kecamatan,kabupaten,luas_desa,jumlah_persil
1659,01061036,Gampong Kampong Blang,Desa,Simpang Tiga,Pidie,788024.1,64
1675,01061055,Gampong Kampong Blang,Desa,Simpang Tiga,Pidie,788207.9,2
4907,01140609,Kuta Padang,Desa,Suka Makmue,Nagan Raya,997683.7,370
4918,01140620,Kuta Padang,Desa,Suka Makmue,Nagan Raya,209763.7,143
7620,02070304,Aek Bange,Desa,Perwakilan Aek Kuasan,ASAHAN,8127650.4,163
7629,02070313,Aek Bange,Desa,Pulau Rakyat,ASAHAN,4726884.7,32
8968,02103607,HUTABARU NANGKA,Desa,HALONGONAN TIMUR,Tapanuli Selatan,13244143.1,118
8975,02103615,HUTABARU NANGKA,Desa,HALONGONAN TIMUR,Tapanuli Selatan,13244143.1,0
9487,02141011,Binasi,Kelurahan,Sorkam Barat,Tapanuli Tengah,2018601.8,63
9488,02141012,Binasi,Desa,Sorkam Barat,Tapanuli Tengah,1145166.1,0


**What Step 1 tells us (read it against the rule-book table above)**

- `jumlah_persil` and `luas_persil` are *derived* (sum of the 8 rights), so keep them out of any "which columns are related" analysis, or use them only as denominators.
- For every right, the count and area columns are zero together, so they carry the same presence/absence information. Treat each right as one concept with two measurements.
- `year present => titled parcel` holds with no exceptions, which supports reading `kosong` as "parcel with no registered right".
- Flood and RDTR areas never exceed the village area, so `banjir_*` and `rdtr_luas` can be turned into shares of `luas_desa`.
- `luas_persil <= luas_desa` is a *soft* rule and is violated often; that is a data-quality signal handled in Step 3.
- The 8-digit `kode` is a reliable hierarchy (province > kab/kota > kecamatan > village); only a handful of rows carry another province's code (flagged in `prov_code_mismatch`).
- Neither names nor codes alone identify a kabupaten. A name can hide a Kabupaten *and* a Kota (e.g. Bandung, Bekasi), and a few `kab_code`s are shared by two names (many pairs look like a regency and the one it was split from, e.g. Malaka/Belu; treat that as a hypothesis to confirm). Build a composite key (province + normalized name + code) before grouping at kabupaten level.
- Casing is inconsistent (UPPER vs Title), so always use the normalized `*_n` columns for names, never the raw ones.